### Import

In [4]:
import os
from pathlib import Path
import pandas as pd 
import numpy as np

try:
    import utils.process_analysis
except ImportError as e:
    print(f"\nSTILL FAILING: {e}")

### Configuration of Paths, Variables

In [5]:
DATA_DIR = "../data/02_processed_combined_data"
OUTPUT_DIR = "../data/04_analysis/h1_results/manipulated_audience_for_revcorr/"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

signals = ['Gaze_X', 'Gaze_Y', 'Pupil_Mean', 'Heart_Rate', 'GSR']
N_SAMPLES = 5000
current_dir = os.getcwd()
CONDITIONS_FILE = os.path.abspath(os.path.join(current_dir, "../data/sub_conditions.csv"))

### Precompute Results

In [6]:
# important: This is for the manipulated movie (PCM)!
pc_subjs = utils.process_analysis.get_subjects_by_condition(CONDITIONS_FILE, target_condition='PCM', exclude_prior_exposure=False) 
results = utils.process_analysis.load_filtered_physiological_data(DATA_DIR, pc_subjs)

summary_stats = []

print("Starting Heavy Computations...")

for signal in signals:
    print(f"Processing {signal}...")
    df = results[signal]
    
    # 1. SPLIT-HALF TIMECOURSE (For visual plotting)
    g1, g2, r_split = utils.process_analysis.get_zscored_split_half(df)
    
    # Save Timecourse to CSV
    tc_df = pd.DataFrame({'Time_Sec': df.index, 'Group_1': g1, 'Group_2': g2})
    tc_df.to_csv(os.path.join(OUTPUT_DIR, f"{signal}_split_timecourse.csv"), index=False)

    # 2. PHASE RANDOMIZATION (The Heavy 5000 permutations)
    # Note: nltools isc expects data to be shaped depending on input, usually (time, subjects)
    stats_phase = utils.process_analysis.isc(df, method='phase_randomize', n_samples=N_SAMPLES, return_null=True)
    
    # Extract values
    obs_isc = stats_phase['isc']
    p_val = stats_phase['p']
    null_dist = stats_phase['null_distribution']
    
    # Save the Null Distribution Array (so we can plot histograms later)
    np.save(os.path.join(OUTPUT_DIR, f"{signal}_null_dist.npy"), null_dist)
    
    # 3. SAVE SUMMARY STATS
    summary_stats.append({
        'Signal': signal,
        'Split_Half_r': r_split,
        'Observed_ISC': obs_isc,
        'Phase_Rand_P': p_val
    })

# Save Master Stats Table
stats_df = pd.DataFrame(summary_stats)
stats_df.to_csv(os.path.join(OUTPUT_DIR, "h1_summary_stats.csv"), index=False)
print(f"All computations saved to '{OUTPUT_DIR}' directory.")

Scanning 194 files against 99 subject IDs...
Skipping sub013 (File: sub013_combined_physio.csv) - Contains NaNs/Missing data
Successfully loaded 96 valid subjects.
Starting Heavy Computations...
Processing Gaze_X...
Processing Gaze_Y...
Processing Pupil_Mean...
Processing Heart_Rate...
Processing GSR...
All computations saved to '../data/04_analysis/h1_results/manipulated_audience_for_revcorr/' directory.
